In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gprofiler import GProfiler
import gseapy as gp

In [ ]:
projdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/multiome/'

### Fail1

In [ ]:
fail1_degs = pd.read_csv(f'{projdir}/csv/degs/fail1_vs_start.csv', sep='\t', index_col=0)
fail1_degs.shape

In [ ]:
fail1_degs.head(20)

In [ ]:
pval_thresh = 1e-20
fc_thresh = 2

mask = (fail1_degs['pvals_adj'] < pval_thresh) & (fail1_degs['logfoldchanges'] > fc_thresh)
fail1_sig = fail1_degs[mask]
fail1_sig.shape

In [ ]:
fail1_sig.head()

In [ ]:
gp = GProfiler(return_dataframe=True)
genes = list(fail1_sig['names'].values)
fail1_results = gp.profile(
    organism='hsapiens',
    query=genes,
    sources=['GO:BP', 'GO:MF', 'GO:CC'],
    no_evidences=False
)

fail1_results.shape

In [ ]:
fail1_results.head()

In [ ]:
select = ['source', 'name', 'p_value', 'intersection_size']
fail1_results[select]

In [ ]:
fail1_results['source'].value_counts()

In [ ]:
fail1_results['name'][:20]

In [ ]:
tmp_fail1_results = fail1_results[fail1_results['source'] == 'GO:CC']
tmp_fail1_results['name'][:20]

In [ ]:
tmp_fail1_results = fail1_results[fail1_results['source'] == 'GO:MF']
tmp_fail1_results['name'][:20]

In [ ]:
fail1_df = fail1_results.sort_values("p_value")
fail1_df = fail1_df[fail1_df["source"] == "GO:BP"]
fail1_top = fail1_df.head(20).copy()
fail1_top["neglog10_p"] = -np.log10(fail1_top["p_value"])
fail1_top.head()

### repeat for start

In [ ]:
start_degs = pd.read_csv(f'{projdir}/csv/degs/start_vs_fail1.csv', sep='\t', index_col=0)
start_degs.shape

In [ ]:
pval_thresh = 1e-20
fc_thresh = 2

mask = (start_degs['pvals_adj'] < pval_thresh) & (start_degs['logfoldchanges'] > fc_thresh)
start_sig = start_degs[mask]
start_sig.shape

In [ ]:
gp = GProfiler(return_dataframe=True)
genes = list(start_sig['names'].values)
start_results = gp.profile(
    organism='hsapiens',
    query=genes,
    sources=['GO:BP', 'GO:MF', 'GO:CC'],
    no_evidences=False
)

start_results.shape

In [ ]:
start_results.head()

In [ ]:
start_results['source'].value_counts()

In [ ]:
start_results['name'][:20]

In [ ]:
tmp_start_results = start_results[start_results['source'] == 'GO:CC']
tmp_start_results['name'][:20]

In [ ]:
tmp_start_results = start_results[start_results['source'] == 'GO:MF']
tmp_start_results['name'][:20]

In [ ]:
start_df = start_results.sort_values("p_value")
start_df = start_df[start_df["source"] == "GO:BP"]
start_top = start_df.head(20).copy()
start_top["neglog10_p"] = -np.log10(start_top["p_value"])
start_top.head()

In [ ]:
fail1_top['cluster'] = 'Fail1'
start_top['cluster'] = 'Start'
# merged = pd.concat([fail1_top, start_top])
# merged.shape

In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    fail1_top["neglog10_p"],            # x-axis
    fail1_top["name"],                  # y-axis (GO term)
    s=fail1_top["intersection_size"]*10 # dot size = gene count
)

plt.xlabel("-log10 p-value")
plt.ylabel("GO Term")
plt.title("GO Enrichment (CT2 vs CT1 Upregulated Genes)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

plt.scatter(
    start_top["neglog10_p"],            # x-axis
    start_top["name"],                  # y-axis (GO term)
    s=start_top["intersection_size"]*10 # dot size = gene count
)

plt.xlabel("-log10 p-value")
plt.ylabel("GO Term")
plt.title("GO Enrichment (CT2 vs CT1 Upregulated Genes)")
plt.tight_layout()
plt.show()

### write top GO terms

In [ ]:
fail1_top.head()

In [ ]:
start_top.head()

In [ ]:
start_go_terms = start_top['native']
start_go = []
for go in start_go_terms:
    go_genes = ','.join(start_top[start_top['native'] == go]['intersections'].values[0])
    start_go.append(go_genes)

In [ ]:
fail1_go_terms = fail1_top['native']
fail1_go = []
for go in fail1_go_terms:
    go_genes = ','.join(fail1_top[fail1_top['native'] == go]['intersections'].values[0])
    fail1_go.append(go_genes)

In [ ]:
start_genes = pd.DataFrame(start_go)
start_genes['cluster'] = 'Start'
start_genes['GO'] = start_go_terms
start_genes['name'] = start_top['name'].values

fail1_genes = pd.DataFrame(fail1_go)
fail1_genes['cluster'] = 'Fail1'
fail1_genes['GO'] = fail1_go_terms
fail1_genes['name'] = fail1_top['name'].values

merged = pd.concat([start_genes, fail1_genes])
merged.columns = ['genes', 'cluster', 'GO', 'name']
merged.shape

In [ ]:
merged.head()

In [ ]:
merged.to_csv(f'{projdir}/csv/degs/fail1/gene_ontology.csv', sep='\t')

### get delayed

### get -log10p p values

In [ ]:
start_go = start_top['name'].values
fail1_go = fail1_top['name'].values

In [ ]:
start_go

In [ ]:
tmp_fail1.index.isin(start_go)f

In [ ]:
tmp_fail1 = fail1_df.copy()
tmp_fail1.index = tmp_fail1['name']
tmp_fail1.reindex(start_go)

In [ ]:
fail1_top.head()['neglog10_p']

In [ ]:
fail1_go.head()

In [ ]:
!date